In [ ]:
def load_efel(dirname):
    path = os.path.join(dirname, 'genetic_alg', 'efel_data')
    pickles = [os.path.join(path,elem) for elem in os.listdir(path) if 'pkl' in elem]
    print(pickles,'pickles')
    assert len(pickles) == 1
    with open(pickles[0],'rb') as f:
        data = pickle.load(f)
    return data

def get_cell_num(dirname):
    cell_num = re.findall(r'\d+', dirname)[-1]
    return cell_num


In [ ]:
# efel_dirs = ['allen_full_11_09_21_488683423',
#              'compare_bbp_full_06_19_23_485574832',
#             #'compare_bbp_full_06_21_23_480351780',]
#              'compare_bbp_full_06_22_23_314822529',
#             'compare_bbp_full_06_22_23_468120757',
#             "compare_bbp_full_06_30_23_314831019",
#             'compare_bbp_full_07_07_23_484559000']

cell_ids = [ '488683423',
            '484559000',
            '480351780',
            '314822529',
            '468120757',
            "314831019",
            '485574832']

models = ['M1_TTPC_NA_HH', 'compare_bbp']


In [ ]:
def df_rows_from_data(data, cell_id):
    rows = []
    for stim in data:
        for feature in data[stim].keys():
            # print(data[stim][feature].keys())
            try:
                rows.append({
                    'cell_id': cell_id,
                    'stim': stim,
                    'feature': feature,
                    'cell_features' : data[stim][feature]['cell_features'],
                 'allen_features' : data[stim][feature]['allen_features'],
                 'compare_bbp_features' : data[stim][feature]['compare_bbp_features'], 
                 'M1_TTPC_NA_HH_features' : data[stim][feature]['M1_TTPC_NA_HH_features']})
            except KeyError:
                if not 'compare_bbp_features' in data[stim][feature]: 
                    # print('skipped :', feature, stim)
                    continue
                rows.append({
                    'cell_id': cell_id,
                    'stim': stim,
                    'feature': feature,
                    'cell_features' : data[stim][feature]['cell_features'],
                 'allen_features' : data[stim][feature]['allen_features'],
                 'compare_bbp_features' : data[stim][feature]['compare_bbp_features'], 
                 'M1_TTPC_NA_HH_features' : [] })
    return rows

In [ ]:
def merge_dicts(data1, data2={}):
    res = copy.deepcopy(data1)
    for key in data2.keys():
        if (key in data2 and key in data1) and type(data2[key]) == dict and type(data1[key])  == dict:
            res[key] = merge_dicts(data2[key],data1[key])
        elif not (key in data2 and key in data1) and key != 'M1_TTPC_NA_HH_features':
            continue
        else:
            res[key] = data2[key]
    return res

def add_efel_model(curr_data, model):
    res = copy.deepcopy(curr_data)
    for stim in curr_data.keys():
        for feature in curr_data[stim].keys():
            res[stim][feature][model + '_features'] = res[stim][feature]['compare_features']
            del res[stim][feature]['compare_features']
    return res
        

In [ ]:
all_cell_efel_data = {}
df_rows = []
for cell_id in cell_ids:
    skip_cell = False
    data = {}
    for model in models:
        candidates = [folder for folder in os.listdir() if model in folder and cell_id in folder]
        if not len(candidates): 
            print('no ', model, cell_id)
            print('skipping for all cells!')
            skip_cell = True
            continue
        curr_e_dir = candidates[-1]
        curr_data = load_efel(curr_e_dir)
        curr_data = add_efel_model(curr_data, model)
        cell_num = get_cell_num(curr_e_dir)
        data = merge_dicts(curr_data, data)
        
    # do not use data where all models are not fit
    if skip_cell: continue
    
    all_cell_efel_data[cell_num] = data
    df_rows += df_rows_from_data(data, cell_num)

    
    # CLEAN OUT STIMS WHERE CELL FEATURES ARE NOT PRESENT
all_cell_efel_data_copy = copy.deepcopy(all_cell_efel_data)
for cell_id in all_cell_efel_data.keys():
    for stim in all_cell_efel_data[cell_id].keys():
        for sf in all_cell_efel_data[cell_id][stim].keys():
            cell_fts = all_cell_efel_data[cell_id][stim][sf]['cell_features']
            gate = len(cell_fts) if isinstance(cell_fts, np.ndarray) else cell_fts
            if not gate: 
                del all_cell_efel_data_copy[cell_id][stim][sf]
all_cell_efel_data =  copy.deepcopy(all_cell_efel_data)

In [ ]:
all_cell_efel_data['468120757'].keys()

In [ ]:
df = pd.DataFrame.from_dict(df_rows)

In [ ]:
df.shape

In [ ]:
df.shape

In [ ]:
df.to_csv('efel_df.csv', index=None)

In [ ]:
with open('efel_data.pkl','wb') as f:
    pickle.dump(all_cell_efel_data, f)

In [ ]:
list(all_cell_efel_data['468120757']['43'])

In [ ]:
df.head(5)

In [ ]:
keep_idxs = []

def bad_row_check(row, feat):
    if type(row[feat]) == np.ndarray and not len(row[feat]):
        return True
    elif  type(row[feat]) ==  np.ndarray  and len(row[feat]):
        return False
    elif  type(row[feat]) ==  np.ndarray  and len(row[feat]) and (row[feat] == np.nan).any():
        return True
    elif row[feat] == None:
        return True
    else:
        return False
    

for idx, row in df.iterrows():
    if bad_row_check(row,'allen_features') or bad_row_check(row,'compare_bbp_features') or bad_row_check(row,'M1_TTPC_NA_HH_features'):
        keep_idxs.append(False)
    else:
        keep_idxs.append(True)

In [ ]:
small_df = df.loc[keep_idxs]

In [ ]:
small_df.apply

In [ ]:
allen_diffs = []
compare_diffs = []
m1_diffs = []
for cell_ft, allen_ft, compare_ft, m1_ft in \
    zip(small_df['cell_features'].values, small_df['allen_features'].values,
        small_df['compare_bbp_features'].values, small_df['M1_TTPC_NA_HH_features'].values):
        if not type(cell_ft) == np.ndarray:
            continue
        
        allen_diffs.append(np.mean(cell_ft) - np.mean(allen_ft))
        compare_diffs.append(np.mean(cell_ft) - np.mean(compare_ft))
        m1_diffs.append(np.mean(cell_ft) - np.mean(m1_ft))
        
allen_diffs_mean = np.nanmean(np.abs(allen_diffs))
compare_diffs_mean = np.nanmean(np.abs(compare_diffs))
m1_diffs_mean = np.nanmean(np.abs(m1_diffs))

# allen_diffs_ci = 1.96 * np.nanstd(np.abs(allen_diffs)) / np.sqrt(len(compare_diffs))
# compare_diffs_ci = 1.96 * np.nanstd(np.abs(compare_diffs)) / np.sqrt(len(compare_diffs))
# m1_diffs_ci = 1.96 * np.nanstd(np.abs(m1_diffs)) / np.sqrt(len(m1_diffs))

# print(f"allen mean ft diff {allen_diffs_mean} +/- {allen_diffs_ci}")
# print(f"compare mean ft diff {compare_diffs_mean} +/- {compare_diffs_ci}")


In [ ]:
np.sqrt(len(compare_diffs))

In [ ]:
from scipy import stats

allen_diffs = np.abs(np.array(allen_diffs)[~np.isnan(allen_diffs)])
compare_diffs = np.abs(np.array(compare_diffs)[~np.isnan(compare_diffs)])
m1_diffs = np.abs(np.array(m1_diffs)[~np.isnan(m1_diffs)])

# allen_diffs_final = np.clip(allen_diffs, -1000,1000)
# compare_diffs_final = np.clip(compare_diffs, -1000,1000)

allen_diffs_final = np.clip(allen_diffs, -100,100)
compare_diffs_final = np.clip(compare_diffs, -100,100)
m1_diffs_final = np.clip(m1_diffs, -100,100)


allen_diffs_final = allen_diffs_final[allen_diffs_final != 0]
compare_diffs_final = compare_diffs_final[compare_diffs_final != 0]
m1_diffs_final = m1_diffs_final[m1_diffs_final != 0]


allen_diffs_final = allen_diffs_final[allen_diffs_final != 1]
compare_diffs_final = compare_diffs_final[compare_diffs_final != 1]
m1_diffs_final = m1_diffs_final[m1_diffs_final != 1]


# stats.ttest_ind(allen_diffs_final,compare_diffs_final)
stats.ttest_ind(m1_diffs_final,compare_diffs_final)

In [ ]:
bins = np.linspace(0,20,200)
plt.hist(allen_diffs_final, alpha=.5, label='allen', bins=bins)
plt.hist(compare_diffs_final, alpha=.5, label='compare', bins=bins)
plt.hist(m1_diffs_final, alpha=.5, label='m1',bins=bins)
plt.legend()

In [ ]:
np.mean(allen_diffs_final),  1.96*np.std(allen_diffs_final) / np.sqrt(len(allen_diffs_final))

In [ ]:
np.mean(compare_diffs_final), 1.96*np.std(compare_diffs_final) / np.sqrt(len(compare_diffs_final))

In [ ]:
np.mean(m1_diffs_final), 1.96*np.std(m1_diffs_final) / np.sqrt(len(m1_diffs_final))